In [1]:
from pathlib import Path

import pandas as pd

from langchain_core.tools import tool

In [2]:
data_path = Path("../data/structured")

employees = pd.read_csv(
    data_path / "employees.csv"
)

attendance = pd.read_csv(
    data_path / "attendance.csv"
)

leave = pd.read_csv(
    data_path / "leave.csv"
)

holidays = pd.read_csv(
    data_path / "holidays.csv"
)

attendance["date"] = pd.to_datetime(
    attendance["date"]
)

leave["start_date"] = pd.to_datetime(
    leave["start_date"]
)

leave["end_date"] = pd.to_datetime(
    leave["end_date"]
)

holidays["date"] = pd.to_datetime(
    holidays["date"]
)

print("Data loaded successfully.")

Data loaded successfully.


In [3]:
kb_path = Path("../data/knowledge_base")

policy_files = list(
    kb_path.glob("*.md")
)

print("Available policies:\n")

for file in policy_files:
    print("-", file.name)

Available policies:

- attendance_policy.md
- working_hours_policy.md
- overtime_policy.md
- sick_leave_policy.md
- leave_policy.md
- remote_work_policy.md


In [4]:
attendance_policy_path = (
    kb_path / "attendance_policy.md"
)

attendance_policy = (
    attendance_policy_path.read_text(
        encoding="utf-8"
    )
)

print(attendance_policy)

# NexaTech GmbH Attendance Policy

## Purpose
This policy defines how employees record working time and office attendance.

## Office Attendance Requirement
Employees assigned to hybrid work are expected to work from the office at least three days per week unless an approved exception applies.

## Attendance Recording
Employees must record their check-in and check-out times using the company attendance system.

## Missing Records
If an employee forgets to record a check-in or check-out, they should submit an attendance correction request to their manager.

## Late Arrival
Employees arriving after 09:00 should notify their manager when required by their team.

## Corrections
Attendance corrections should normally be submitted within five working days.

## Business Travel
Approved business travel is considered a working day but does not automatically count as an office day for the hybrid attendance requirement.



In [5]:
remote_policy_path = (
    kb_path / "remote_work_policy.md"
)

remote_policy = (
    remote_policy_path.read_text(
        encoding="utf-8"
    )

)

print(remote_policy)

# NexaTech GmbH Remote Work Policy

## Eligibility
Full-time employees may work remotely if their role permits remote work.

## Weekly Remote Work Limit
Employees may normally work remotely for a maximum of two regular working days per week.

## Office Requirement
Employees assigned to hybrid work are expected to work from the office for at least three days per week.

## Core Hours
Employees working remotely must remain available between 09:00 and 15:00.

## Exceptions
Managers may approve temporary exceptions for business, family, or other justified circumstances.

## Attendance
Remote work days count as working days but do not count toward the minimum office attendance requirement.



In [6]:
POLICY_CONFIG = {
    "minimum_office_days_per_week": 3,
    "maximum_home_days_per_week": 2
}

print(POLICY_CONFIG)

{'minimum_office_days_per_week': 3, 'maximum_home_days_per_week': 2}


In [7]:
def calculate_weekly_attendance(
    employee_id,
    start_date,
    end_date
):
    """
    Calculate office and home-office attendance
    for each week in a specified period.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ].copy()

    if records.empty:
        return pd.DataFrame()

    records["week"] = (
        records["date"]
        .dt.to_period("W-MON")
        .astype(str)
    )

    weekly = (
        records
        .groupby("week")
        .agg(
            office_days=(
                "location",
                lambda x: (x == "office").sum()
            ),
            home_days=(
                "location",
                lambda x: (x == "home").sum()
            ),
            business_trip_days=(
                "location",
                lambda x: (x == "business_trip").sum()
            ),
            sick_days=(
                "status",
                lambda x: (x == "sick").sum()
            ),
            leave_days=(
                "status",
                lambda x: (x == "leave").sum()
            )
        )
        .reset_index()
    )

    return weekly

In [8]:
weekly = calculate_weekly_attendance(
    employee_id="E0001",
    start_date="2026-08-01",
    end_date="2026-08-31"
)

display(weekly)

,week,office_days,home_days,business_trip_days,sick_days,leave_days
0,2026-07-28/2026-08-03,1,0,0,0,0
1,2026-08-04/2026-08-10,4,0,1,0,0
2,2026-08-11/2026-08-17,3,2,0,0,0
3,2026-08-18/2026-08-24,3,2,0,0,0
4,2026-08-25/2026-08-31,4,1,0,0,0


In [12]:
def calculate_remote_work_compliance(
    employee_id,
    start_date,
    end_date
):
    """
    Calculate whether an employee met the
    weekly office attendance requirement.
    """

    weekly = calculate_weekly_attendance(
        employee_id,
        start_date,
        end_date
    )

    if weekly.empty:
        return {
            "employee_id": employee_id,
            "compliant": None,
            "reason": "No attendance data found."
        }

    minimum_office_days = (
        POLICY_CONFIG[
            "minimum_office_days_per_week"
        ]
    )

    weekly["compliant"] = (
        weekly["office_days"]
        >= minimum_office_days
    )

    failed_weeks = weekly[
        ~weekly["compliant"]
    ]

    return {
        "employee_id": employee_id,
        "period":
            f"{start_date} to {end_date}",
        "minimum_office_days_per_week":
            minimum_office_days,
        "weeks_checked":
            len(weekly),
        "weeks_compliant":
            int(weekly["compliant"].sum()),
        "weeks_not_compliant":
            int((~weekly["compliant"]).sum()),
        "compliant":
            bool(weekly["compliant"].all()),
        "failed_weeks":
            failed_weeks["week"].tolist()
    }

In [15]:
result = calculate_remote_work_compliance(
    employee_id="E0001",
    start_date="2026-08-01",
    end_date="2026-08-31"
)

print(result)

{'employee_id': 'E0001', 'period': '2026-08-01 to 2026-08-31', 'minimum_office_days_per_week': 3, 'weeks_checked': 5, 'weeks_compliant': 4, 'weeks_not_compliant': 1, 'compliant': False, 'failed_weeks': ['2026-07-28/2026-08-03']}


In [16]:
@tool
def check_remote_work_compliance(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Check whether an employee complied with the
    company's weekly office attendance requirement.
    """

    result = calculate_remote_work_compliance(
        employee_id,
        start_date,
        end_date
    )

    if result["compliant"] is None:
        return (
            f"No attendance data found "
            f"for {employee_id}."
        )

    status = (
        "COMPLIANT"
        if result["compliant"]
        else "NOT COMPLIANT"
    )

    response = (
        f"Employee: {result['employee_id']}\n"
        f"Period: {result['period']}\n"
        f"Required office days per week: "
        f"{result['minimum_office_days_per_week']}\n"
        f"Weeks checked: {result['weeks_checked']}\n"
        f"Compliant weeks: "
        f"{result['weeks_compliant']}\n"
        f"Non-compliant weeks: "
        f"{result['weeks_not_compliant']}\n"
        f"Overall status: {status}"
    )

    if result["failed_weeks"]:
        response += (
            "\nWeeks below the office requirement:\n"
            + "\n".join(
                result["failed_weeks"]
            )
        )

    return response

In [17]:
print(
    check_remote_work_compliance.invoke({
        "employee_id": "E0001",
        "start_date": "2026-08-01",
        "end_date": "2026-08-31"
    })
)

Employee: E0001
Period: 2026-08-01 to 2026-08-31
Required office days per week: 3
Weeks checked: 5
Compliant weeks: 4
Non-compliant weeks: 1
Overall status: NOT COMPLIANT
Weeks below the office requirement:
2026-07-28/2026-08-03


In [20]:
def calculate_working_hours_compliance(
    employee_id,
    start_date,
    end_date
):
    """
    Compare an employee's recorded working hours
    against their contracted weekly hours.
    """

    employee_result = employees[
        employees["employee_id"] == employee_id
    ]

    if employee_result.empty:
        return {
            "employee_id": employee_id,
            "compliant": None,
            "reason": "Employee not found."
        }

    employee = employee_result.iloc[0]

    weekly_contract_hours = float(
        employee["weekly_hours"]
    )

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ].copy()

    records = records[
        records["check_in"].notna() &
        records["check_out"].notna() &
        (records["check_in"] != "") &
        (records["check_out"] != "")
    ]

    if records.empty:
        return {
            "employee_id": employee_id,
            "compliant": None,
            "reason": "No complete attendance records."
        }

    records["check_in_time"] = pd.to_datetime(
        records["check_in"],
        format="%H:%M"
    )

    records["check_out_time"] = pd.to_datetime(
        records["check_out"],
        format="%H:%M"
    )

    records["hours"] = (
        records["check_out_time"]
        - records["check_in_time"]
    ).dt.total_seconds() / 3600

    total_hours = records["hours"].sum()

    number_of_weeks = max(
        (end - start).days / 7,
        1
    )

    expected_hours = (
        weekly_contract_hours
        * number_of_weeks
    )

    return {
        "employee_id": employee_id,
        "contracted_weekly_hours":
            weekly_contract_hours,
        "recorded_hours":
            round(total_hours, 2),
        "expected_hours":
            round(expected_hours, 2),
        "compliant":
            total_hours >= expected_hours
    }

In [21]:
@tool
def check_working_hours_compliance(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Check whether an employee's recorded working hours
    meet their contracted working hours for a period.
    """

    result = calculate_working_hours_compliance(
        employee_id,
        start_date,
        end_date
    )

    if result["compliant"] is None:
        return result["reason"]

    status = (
        "COMPLIANT"
        if result["compliant"]
        else "BELOW EXPECTED HOURS"
    )

    return (
        f"Employee: {result['employee_id']}\n"
        f"Contracted weekly hours: "
        f"{result['contracted_weekly_hours']}\n"
        f"Recorded hours: "
        f"{result['recorded_hours']}\n"
        f"Expected hours: "
        f"{result['expected_hours']}\n"
        f"Status: {status}"
    )

In [22]:
@tool
def check_working_hours_compliance(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Check whether an employee's recorded working hours
    meet their contracted working hours for a period.
    """

    result = calculate_working_hours_compliance(
        employee_id,
        start_date,
        end_date
    )

    if result["compliant"] is None:
        return result["reason"]

    status = (
        "COMPLIANT"
        if result["compliant"]
        else "BELOW EXPECTED HOURS"
    )

    return (
        f"Employee: {result['employee_id']}\n"
        f"Contracted weekly hours: "
        f"{result['contracted_weekly_hours']}\n"
        f"Recorded hours: "
        f"{result['recorded_hours']}\n"
        f"Expected hours: "
        f"{result['expected_hours']}\n"
        f"Status: {status}"
    )

In [23]:
print(
    check_working_hours_compliance.invoke({
        "employee_id": "E0001",
        "start_date": "2026-08-01",
        "end_date": "2026-08-31"
    })
)

Employee: E0001
Contracted weekly hours: 40.0
Recorded hours: 148.65
Expected hours: 171.43
Status: BELOW EXPECTED HOURS


In [24]:
compliance_tools = [
    check_remote_work_compliance,
    check_working_hours_compliance
]

for tool in compliance_tools:
    print("-", tool.name)

- check_remote_work_compliance
- check_working_hours_compliance
